# TorchRL

Let's start our first tutorial! Here we'll go over how to create *Environments* and how *agents* can take actions in these environments.

* Note: this notebook was modified from the official tutorials provided by torchRL, you can check it out here: https://docs.pytorch.org/rl/stable/tutorials


# Install TorchRL

First let's install torchrl and tensordict.


In [ ]:
#Option 1: easy
#If you are running this from colab you can run:
#!pip install tensordict
#!pip install torchrl

#Option 2: harder
#if you are using your own compputer you should create a virtual environment, activate it, and then use pip to install the needed 
#conda create -n torchrl-env python=3.11 -y
#conda activate torchrl-env
#pip install torchrl tensordict torch torchvision 'gymnasium[all]'
#Note: for GPU support this might be a little different, and might vary by device!

# Environments-agents

Let's start by creating an environment. Let's create the pendulum environment. You can see more about this environment (and others) here: https://gymnasium.farama.org/environments/classic_control/pendulum/

In [ ]:
from torchrl.envs import GymEnv

env = GymEnv("Pendulum-v1")


Environments in TorchRL have two crucial methods: 

1. *reset()*, which initiates an episode, and 
2. *step()*, which executes an action selected by the actor. 

In TorchRL, environment methods read and write TensorDict instances. Essentially, TensorDict is a generic key-based data carrier for tensors. The benefit of using TensorDict over plain tensors is that it enables us to handle simple and complex data structures interchangeably. As our function signatures are very generic, it eliminates the challenge of accommodating different data formats. In simpler terms, after this brief tutorial, you will be capable of operating on both simple and highly complex environments, as their user-facing API is identical and simple!

Let’s create a new environment, reset it, and see what a tensordict instance looks like:

In [ ]:
data_out = env.reset()
print(data_out)

The tensorDict above shows what keys it holds: 

* done: did the agent reach the end of an episode -- size [1], type [boolean]
* observation: what does the agent know (stimuli) -- size [3], type [float32] 
* terminated: did the agent-environment interaction end naturally (reached the end of a maze) -- size [1], type [boolean]
* truncated: did the agent-environment interaction end un-naturally (time ran out) -- size [1], type [boolean]

Each has a size and a data type.

Let's see what values are in the observation key.

In [ ]:
data_out["observation"]

These are the observations of the agent (a little pendulum) and represent the x-y coordinates of the pendulum’s free end and its angular velocity.

Now let’s take a random action in the action space. For the pendulum environment we can take action by applying some amount of torque on the free end of the pendulum (min -2.0, max 2.0).

Let's apply a random action.

In [ ]:
data_out_with_action = env.rand_action(data_out)
print(data_out_with_action)

This tensordict has the same structure as the one obtained from env with an additional "action" entry. You can access the action easily, like you would do with a regular dictionary:

In [ ]:
print(data_out_with_action["action"])

That's the amount of torque you'd like to apply to the pendulum.

We now need to pass this action to the environment. We’ll be passing the entire tensordict to the step method, since there might be more than one tensor to be read in more advanced cases like Multi-Agent RL or stateless environments:

In [ ]:
stepped_data = env.step(data_out_with_action)
print(stepped_data)

Again, this new tensordict is identical to the previous one except for the fact that it has a "next" entry (itself a tensordict!) containing the observation, reward and done state resulting from our action.

We call this format TED, for [TorchRL Episode Data format](https://docs.pytorch.org/rl/stable/reference/data_datasets.html#ted-format). It is the ubiquitous way of representing data in the library, both dynamically like here, or statically with offline datasets.

The last bit of information you need to run a rollout in the environment is how to bring that "next" entry at the root to perform the next step. TorchRL provides a dedicated step_mdp() function that does just that: it filters out the information you won’t need and delivers a data structure corresponding to your observation after a step in the Markov Decision Process, or MDP.

In [ ]:
from torchrl.envs import step_mdp

data = step_mdp(stepped_data)
print(data)

### Environment rollouts

Writing down those three steps:

* computing an action, 
* making a step, 
* moving in the MDP

can be a bit tedious and repetitive. Fortunately, TorchRL provides a nice rollout() function that allows you to run them in a closed loop at will:

In [ ]:
rollout_data = env.rollout(max_steps=10)
print(rollout_data)

This data looks pretty much like the stepped_data above with the exception of its batch-size, which now equates the number of steps we provided through the max_steps argument. The magic of tensordict doesn’t end there: if you’re interested in a single transition of this environment, you can index the tensordict like you would index a tensor:

In [ ]:
transition = rollout_data[3]
print(transition)

TensorDict will automatically check if the index you provided is a key (in which case we index along the key-dimension) or a spatial index like here.

Executed as such (without a policy), the rollout method may seem rather useless: it just runs random actions. If a policy is available, it can be passed to the method and used to collect data.

Nevertheless, it can useful to run a naive, policyless rollout at first to check what is to be expected from an environment at a glance.

To appreciate the versatility of TorchRL’s API, consider the fact that the rollout method is universally applicable. It functions across all use cases, whether you’re working with a single environment like this one, multiple copies across various processes, a multi-agent environment, or even a stateless version of it!

**Things to try**

When creating the environment at the top of this notebook, try changing the environment to "CartPole-v1". Walk through the code and see how the tensodict has changed.

Here's a link to the cartpole environment: https://gymnasium.farama.org/environments/classic_control/cart_pole/

### Transforming an environment

Most of the time, you’ll want to modify the output of the environment to better suit your requirements. For example, you might want to monitor the number of steps executed since the last reset, resize images, or stack consecutive observations together.

In this section, we’ll examine a simple transform, the StepCounter transform. The complete list of transforms can be found [here](https://docs.pytorch.org/rl/stable/reference/envs_transforms.html#transforms).

The transform is integrated with the environment through a TransformedEnv:

In [ ]:
from torchrl.envs import StepCounter, TransformedEnv

transformed_env = TransformedEnv(env, StepCounter(max_steps=10))
rollout = transformed_env.rollout(max_steps=100)
print(rollout)

As you can see, our environment now has one more entry, "step_count" that tracks the number of steps since the last reset. Given that we passed the optional argument max_steps=10 to the transform constructor, we also truncated the trajectory after 10 steps (not completing a full rollout of 100 steps like we asked with the rollout call). We can see that the trajectory was truncated by looking at the truncated entry:

In [ ]:
print(rollout["next", "truncated"])

### Visualizing our environments

Let's use transforms to see an animation of our agents performing in these environments. 

Specfically, if we set from_pixels to True so that the rollout will contain images of the environment/agent, and pixels_only to False so we can see the observations (x,y,torque) and the image (pixels).

In [ ]:

#Note: you might have to install matplotlib
#!pip install matplotlib
from torchrl.envs.libs.gym import GymEnv
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import imageio
from IPython.display import Image
from torchrl.envs.transforms import RewardSum
from torchrl.envs import TransformedEnv, Compose

#create your environment
env = GymEnv("Pendulum-v1", from_pixels=True, pixels_only=False)

#Transform your environment: add in step counter and reward sums per episode
env = TransformedEnv(env,
                     Compose(
        StepCounter(),
        RewardSum()
        )
    )

#run a rollout: i.e., random policy with 200 steps (the agent won't do very well! :-) )
rollout = env.rollout(max_steps=200)

#End the environment
env.close()

#grab all the frames from the rollout
frames = rollout["pixels"].numpy()  

#save the images as a gif
imageio.mimsave("rollout.gif", frames, fps=30)

#display the gif
Image("rollout.gif")   



Let's see what kinds of rewards we were getting.

In [ ]:
rollout["next","reward"]

Let's plot these rewards!

In [ ]:
import matplotlib.pyplot as pyplot

pyplot.scatter(x=rollout["step_count"], y=rollout["next","reward"])

**Things to try**

Try this with cartpole. 

> Why does the animation stop so quickly?

>> Try doing the rollout with: break_when_any_done=False

> Why do we always get the same reward? 

>> Try calculating the rewards accumulated for each episode by using the done values

In [ ]:
#get dones: when did episodes end
#done = ? .squeeze(-1)  # shape: [200]

#use dones to select cummulative rewards: get reward sums at the end of the episode
#episode_rewards = rollout["next", "?"][?]

#plot it
#pyplot.scatter(x=range(len(episode_rewards)), y=episode_rewards)

# Next steps:

Let's look at how we can start using RL algorithms to make our agents smarter.

* Read up more about [TorchRL Episod Data](https://docs.pytorch.org/rl/stable/reference/data_datasets.html#ted-format) 
* Read up more about [transforms](https://docs.pytorch.org/rl/stable/reference/envs_transforms.html#transforms) 
* Look at more [environments](https://gymnasium.farama.org/) 